# Lesson 1 — The Italian Game: b4 Habit (as White)
**Project 1500 | Priority #1**

All statistics computed fresh from game files on every run. No hardcoded results.
Board diagrams are illustrative example lines, clearly labelled as such.

In [ ]:
import chess
import chess.pgn
import chess.svg
import chess.engine
import asyncio
import sys
import json
import glob
import io
import os
import re
import numpy as np
from collections import Counter
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

USERNAME   = 'jf4bes'
EXCLUDE    = {'2025_01', '2025_02', '2025_03'}
BOARD_SIZE = 380

# Windows + Jupyter fix: switch to ProactorEventLoop so subprocess (Stockfish) works
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

def board_after(moves_san):
    """Apply SAN moves to a fresh board — illustrative diagrams only."""
    board = chess.Board()
    for san in moves_san:
        board.push_san(san)
    return board

def show_board(board, arrows=None, caption='', size=BOARD_SIZE):
    svg = chess.svg.board(board, arrows=arrows or [], size=size)
    if caption:
        display(HTML(
            f'<p style="font-family:monospace;font-size:13px;'
            f'margin:4px 0 8px 0;color:#aaa">{caption}</p>'
        ))
    display(SVG(data=svg))

## Step 1 — Load all rapid games as white (April 2025+)

In [ ]:
def load_rapid_games_as_white(username, games_dir='games', exclude=EXCLUDE):
    files = sorted(
        glob.glob(f'{games_dir}/2025_*.json') +
        glob.glob(f'{games_dir}/2026_*.json')
    )
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') != 'rapid': continue
            white = g.get('white', {})
            if white.get('username', '').lower() != username.lower(): continue
            result = white.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({'outcome': outcome, 'pgn': g.get('pgn', ''), 'eco_url': eco_url})
    return games

all_white_games = load_rapid_games_as_white(USERNAME)
print(f'Loaded {len(all_white_games)} rapid games as white')

## Step 2 — Detect Italian structure, classify b4 vs no-b4

**Italian structure:** white has a pawn on both **e4** and **c3** simultaneously.
Detected from the position — no opening-name filter.

For each Italian game: did white play **b4** at any point in the first 30 half-moves?

In [ ]:
def is_italian_structure(board):
    """True when white has pawns on both e4 and c3."""
    e4 = board.piece_at(chess.E4)
    c3 = board.piece_at(chess.C3)
    return (
        e4 and e4.piece_type == chess.PAWN and e4.color == chess.WHITE and
        c3 and c3.piece_type == chess.PAWN and c3.color == chess.WHITE
    )

def eco_short_label(url, words=5):
    if '/openings/' not in url: return 'Unknown'
    return ' '.join(url.split('/openings/')[-1].split('-')[:words])

b4_games    = []
no_b4_games = []

for g in all_white_games:
    try:
        game  = chess.pgn.read_game(io.StringIO(g['pgn']))
        board = game.board()
        moves = list(game.mainline_moves())
        italian_reached = False
        played_b4       = False
        for i, move in enumerate(moves[:30]):
            if i % 2 == 0:
                if is_italian_structure(board): italian_reached = True
                if italian_reached and board.san(move) == 'b4':
                    played_b4 = True
                    b4_games.append({'outcome': g['outcome'], 'label': eco_short_label(g['eco_url']), 'pgn': g['pgn']})
                    break
            board.push(move)
        if italian_reached and not played_b4:
            no_b4_games.append({'outcome': g['outcome']})
    except Exception:
        pass

b4_counts    = Counter(g['outcome'] for g in b4_games)
no_b4_counts = Counter(g['outcome'] for g in no_b4_games)
b4_total     = len(b4_games)
no_b4_total  = len(no_b4_games)
b4_wr        = 100 * b4_counts['win']    / b4_total    if b4_total    else 0
no_b4_wr     = 100 * no_b4_counts['win'] / no_b4_total if no_b4_total else 0

print(f"Italian games — b4 played:     {b4_total:3d}  W:{b4_counts['win']} L:{b4_counts['loss']} D:{b4_counts['draw']}  Win%: {b4_wr:.1f}%")
print(f"Italian games — b4 NOT played: {no_b4_total:3d}  W:{no_b4_counts['win']} L:{no_b4_counts['loss']} D:{no_b4_counts['draw']}  Win%: {no_b4_wr:.1f}%")
print(f"\nCost of playing b4: {no_b4_wr - b4_wr:+.1f} percentage points")

## Step 3 — One habit, many opening labels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, label, counts, total, col in [
    (axes[0], f'Played b4\n({b4_total} games)',         b4_counts,    b4_total,    '#c0392b'),
    (axes[1], f'Did NOT play b4\n({no_b4_total} games)', no_b4_counts, no_b4_total, '#27ae60'),
]:
    wr = 100 * counts['win'] / total
    ax.bar(['Win','Loss','Draw'], [counts['win'],counts['loss'],counts['draw']],
           color=[col,'#666','#999'], edgecolor='white')
    ax.set_title(f"{label}\n{wr:.1f}% win rate")
    ax.set_ylabel('Games')
plt.suptitle('Italian Game (e4+c3 detected): b4 vs no b4  —  April 2025+', fontsize=13)
plt.tight_layout()
plt.show()

label_counts = Counter(g['label'] for g in b4_games)
print(f"b4 spans {len(label_counts)} different Chess.com opening labels:")
for lbl, n in label_counts.most_common():
    print(f"  {n:3d}  {lbl}")

## Step 4 — Stockfish: is b4 the mistake, or the follow-up?

Win rate data shows b4 *correlates* with losses — but correlation is not causation. It could mean:

1. **b4 itself is objectively bad** — eval drops the moment it's played
2. **The follow-up is the problem** — b4 is borderline, but a4 chasing the bishop is where you lose the thread
3. **b4 is a symptom** — players who play b4 are also delaying castling, so the losses come from the whole approach

We evaluate each b4 game at three stages and measure the centipawn change:
- **Before b4** — baseline (what was the position worth before the move?)
- **After b4** — did b4 itself cost material?
- **After bishop retreats** — did the follow-up make things worse?

In [ ]:
STOCKFISH_PATH = r'E:\Github\Chesslatro\chesslatro\stockfish\stockfish.exe'
EVAL_DEPTH     = 12
MAX_GAMES      = 60
EXTREME_CP     = 500

def get_eval_cp(engine, board, depth=EVAL_DEPTH):
    """Centipawn eval from WHITE's perspective. Mate = ±10000."""
    info  = engine.analyse(board, chess.engine.Limit(depth=depth))
    score = info['score'].white()
    if score.is_mate():
        return 10000 if score.mate() > 0 else -10000
    return score.score()

# Store evals as per-game dicts so deltas are always computed from the same game
game_evals = []  # [{'ev0': int, 'ev1': int, 'ev2': int (optional)}, ...]

if not os.path.exists(STOCKFISH_PATH):
    print(f'Stockfish not found at:\n  {STOCKFISH_PATH}')
    game_evals = None
else:
    engine   = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
    analyzed = 0
    skipped  = 0

    for g in b4_games:
        if analyzed >= MAX_GAMES: break
        try:
            game  = chess.pgn.read_game(io.StringIO(g['pgn']))
            board = game.board()
            moves = list(game.mainline_moves())
            italian_reached = False

            for i, move in enumerate(moves[:30]):
                if i % 2 == 0:
                    if is_italian_structure(board): italian_reached = True
                    if italian_reached and board.san(move) == 'b4':
                        ev0 = get_eval_cp(engine, board)
                        if abs(ev0) > EXTREME_CP:
                            skipped += 1
                            break

                        board.push(move)
                        ev1 = get_eval_cp(engine, board)

                        rec = {'ev0': ev0, 'ev1': ev1}

                        # ev2: only if black's next move is a bishop retreat
                        if i + 1 < len(moves):
                            resp_san = board.san(moves[i + 1])
                            if resp_san in ('Bb6', 'Ba5', 'Bd4', 'Bd6', 'Be7'):
                                board.push(moves[i + 1])
                                rec['ev2'] = get_eval_cp(engine, board)

                        game_evals.append(rec)
                        analyzed += 1
                        break
                board.push(move)
        except Exception:
            pass

    engine.quit()

    n_all  = len(game_evals)
    n_bb6  = sum(1 for r in game_evals if 'ev2' in r)

    print(f'Analyzed {analyzed} games  (skipped {skipped} already-decided positions)')
    print(f'Bishop retreat observed in {n_bb6} of {n_all} games')
    print()

    avg_ev0 = np.mean([r['ev0'] for r in game_evals])
    avg_ev1 = np.mean([r['ev1'] for r in game_evals])
    # delta_b4: paired difference (same games for ev0 and ev1)
    delta_b4 = np.mean([r['ev1'] - r['ev0'] for r in game_evals])

    print(f'  Before b4 (n={n_all}):                {avg_ev0:+.0f} cp')
    print(f'  After  b4 (n={n_all}):                {avg_ev1:+.0f} cp')
    print(f'  Delta at b4:                     {delta_b4:+.0f} cp  (negative = b4 cost white material)')

    if n_bb6 > 0:
        # delta_bb6: only from games where bishop actually retreated (paired ev1 and ev2)
        avg_ev1_bb6  = np.mean([r['ev1'] for r in game_evals if 'ev2' in r])
        avg_ev2_bb6  = np.mean([r['ev2'] for r in game_evals if 'ev2' in r])
        delta_bb6    = np.mean([r['ev2'] - r['ev1'] for r in game_evals if 'ev2' in r])
        print()
        print(f'  After b4, before retreat (n={n_bb6}):  {avg_ev1_bb6:+.0f} cp')
        print(f'  After bishop retreats    (n={n_bb6}):  {avg_ev2_bb6:+.0f} cp')
        print(f'  Delta after retreat:             {delta_bb6:+.0f} cp  (negative = retreat good for black)')

In [ ]:
if game_evals:
    n_all = len(game_evals)
    n_bb6 = sum(1 for r in game_evals if 'ev2' in r)

    # Build paired plot data
    plot_data = [
        ('Before b4',            [r['ev0'] for r in game_evals],              n_all),
        ('After b4',             [r['ev1'] for r in game_evals],              n_all),
        ('After bishop\\nretreats', [r['ev2'] for r in game_evals if 'ev2' in r], n_bb6),
    ]
    plot_data = [(lbl, evs, n) for lbl, evs, n in plot_data if evs]

    xlbls  = [d[0] for d in plot_data]
    ys     = [np.mean(d[1]) for d in plot_data]
    yerrs  = [np.std(d[1])  for d in plot_data]
    ns     = [d[2]          for d in plot_data]
    colors = ['#888888', '#c0392b', '#e67e22'][:len(plot_data)]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(xlbls, ys, yerr=yerrs, color=colors, capsize=6, edgecolor='white', width=0.5)
    ax.axhline(0, color='white', linestyle='--', linewidth=1, alpha=0.4)
    ax.set_ylabel('Centipawns (white perspective — positive = white better)')
    ax.set_title(f'Stockfish eval at each stage  (depth {EVAL_DEPTH}, error bars = 1 std dev)')

    for bar, val, n in zip(bars, ys, ns):
        offset = 15 if val >= 0 else -30
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + offset,
                f'{val:+.0f}cp\\n(n={n})', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()

    # Interpretation — all deltas now properly paired
    delta_b4 = np.mean([r['ev1'] - r['ev0'] for r in game_evals])
    print()
    if abs(delta_b4) < 20:
        print(f'b4 itself: {delta_b4:+.0f}cp — roughly neutral (< 20cp change).')
        print('The damage comes from the FOLLOW-UP, not b4 alone.')
    else:
        print(f'b4 itself: {delta_b4:+.0f}cp — b4 is objectively a mistake.')

    if n_bb6 > 0:
        delta_bb6 = np.mean([r['ev2'] - r['ev1'] for r in game_evals if 'ev2' in r])
        if delta_bb6 < -10:
            print(f'Bishop retreat: {delta_bb6:+.0f}cp — retreat is good for black, compounds the damage.')
        elif delta_bb6 > 10:
            print(f'Bishop retreat: {delta_bb6:+.0f}cp — retreat slightly helps white (bishop was bad on c5).')
        else:
            print(f'Bishop retreat: {delta_bb6:+.0f}cp — roughly neutral.')

## Step 5 — Board diagrams *(illustrative example lines)*

These diagrams show what the positions look like — fixed example lines,
not derived from the statistical analysis above.

In [ ]:
EXAMPLE_ITALIAN = ['e4','e5','Nf3','Nc6','Bc4','Bc5','c3','Nf6','d3','O-O']

show_board(
    board_after(EXAMPLE_ITALIAN),
    arrows=[
        chess.svg.Arrow(chess.B2, chess.B4, color='#c0392b'),
        chess.svg.Arrow(chess.E1, chess.G1, color='#27ae60'),
        chess.svg.Arrow(chess.B1, chess.D2, color='#2980b9'),
    ],
    caption='[ILLUSTRATIVE] After 5.d3 O-O — Red=b4 (avoid) | Green=O-O (castle first) | Blue=Nbd2 (after castling)'
)

In [ ]:
show_board(
    board_after(EXAMPLE_ITALIAN + ['b4','Bb6','a4','a6']),
    arrows=[
        chess.svg.Arrow(chess.A4, chess.A4, color='#c0392b'),
        chess.svg.Arrow(chess.B4, chess.B4, color='#c0392b'),
    ],
    caption='[ILLUSTRATIVE] After b4 Bb6 a4 a6 — 3 moves wasted, Nb1 undeveloped, king uncastled'
)

In [ ]:
show_board(
    board_after(EXAMPLE_ITALIAN + ['O-O','d6','Nbd2']),
    arrows=[
        chess.svg.Arrow(chess.H1, chess.E1, color='#27ae60'),
        chess.svg.Arrow(chess.D3, chess.D4, color='#2980b9'),
    ],
    caption='[ILLUSTRATIVE] After O-O d6 Nbd2 — king safe, Re1 next (green), then d4 (blue)'
)

## Step 6 — Summary

In [ ]:
print('ITALIAN GAME — SUMMARY')
print('=' * 50)
print(f'\nItalian games with b4:    {b4_total} games, {b4_wr:.1f}% win rate')
print(f'Italian games without b4: {no_b4_total} games, {no_b4_wr:.1f}% win rate')
print(f'Cost of playing b4:       {no_b4_wr - b4_wr:+.1f} percentage points')
print(f'\nb4 found across {len(label_counts)} different Chess.com opening labels')
print()
print('RULES:')
print('  1. c3 before d3 — always')
print('  2. Castle (O-O) before any structural plan')
print('  3. Never play b4')
print('  4. After castling: Nbd2, then Re1, then d4')